## Challenge 2


Run on Google Colab:

[![Open In Collab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/drive/1u76LRcNfNdp76sfAId2bcy2JyreLoExE?usp=sharing)

In [ ]:
!nvidia-smi -L

In [ ]:
!pip install trl==0.11.0 transformers==4.45.0

In [ ]:
!pip -q install "datasets>=2.19" "accelerate>=0.33" sentencepiece

In [ ]:
import os, json, math, random, time
from dataclasses import dataclass
from typing import List, Dict

import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
from google.colab import files
uploaded = files.upload()  # select preference_data.jsonl

DATA_PATH = "preference_data.jsonl"
for fname, data in uploaded.items():
    with open(DATA_PATH, 'wb') as f:
        f.write(data)
print('Ready:', DATA_PATH)


In [ ]:
def load_jsonl(path: str):
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            r = json.loads(line)
            if all(k in r for k in ['prompt', 'chosen', 'rejected']):
                rows.append({'prompt': r['prompt'], 'chosen': r['chosen'], 'rejected': r['rejected']})
    return rows

pairs = load_jsonl(DATA_PATH)
print('Loaded pairs:', len(pairs))
print(pairs[0].keys() if pairs else 'No data')


In [ ]:
random.seed(0)
random.shuffle(pairs)

split = int(0.8 * len(pairs))
train_pairs = pairs[:split]
test_pairs  = pairs[split:] if split < len(pairs) else pairs[:2]  # ensure non-empty

train_ds = Dataset.from_list(train_pairs)
test_ds  = Dataset.from_list(test_pairs)

len(train_ds), len(test_ds)

In [ ]:
MODEL_NAME = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"  # required for batched causal generation

# Keep a frozen base model for pre/post-PPO evaluation comparison
base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
base_model.eval()
for p in base_model.parameters():
    p.requires_grad_(False)

print(f"Loaded {MODEL_NAME}")


In [ ]:
def tokenize_prompt_and_response(prompt: str, response: str, max_len=256):
    # We train on prompt + response, but only compute loss on response tokens
    full = prompt + "\n" + response
    enc_full = tokenizer(full, return_tensors="pt", truncation=True, max_length=max_len, padding=False)
    enc_prompt = tokenizer(prompt + "\n", return_tensors="pt", truncation=True, max_length=max_len, padding=False)
    return enc_full, enc_prompt["input_ids"].shape[1]

def logprobs_from_logits(logits, labels):
    # logits: [B, T, V], labels: [B, T]
    logp = F.log_softmax(logits, dim=-1)
    # gather token logprobs
    tok_logp = torch.gather(logp, dim=-1, index=labels.unsqueeze(-1)).squeeze(-1)
    return tok_logp

@torch.no_grad()
def kl_tokenwise(policy_logits, ref_logits):
    # KL(policy || ref) tokenwise
    p = F.softmax(policy_logits, dim=-1)
    logp = F.log_softmax(policy_logits, dim=-1)
    logr = F.log_softmax(ref_logits, dim=-1)
    return torch.sum(p * (logp - logr), dim=-1)  # [B, T]




---



Step 1: training reward model

In [ ]:
from trl import RewardTrainer, RewardConfig
from transformers import AutoModelForSequenceClassification
from trl.models import AutoModelForCausalLMWithValueHead
from trl import PPOConfig, PPOTrainer

# Key fix: truncate from the LEFT so the VA turn at the END is always preserved
# The distinguishing signal ('(silent)' vs verbose response) is the final line.
# Right-truncation cuts it off entirely, making chosen/rejected look identical to the RM.
tokenizer.truncation_side = "left"
MAX_LEN_RM = 256

rm_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=1).to(device)
rm_model.config.pad_token_id = tokenizer.pad_token_id

def preprocess_rm(example):
    chosen_enc   = tokenizer(example["prompt"] + "\n" + example["chosen"],
                             truncation=True, max_length=MAX_LEN_RM)
    rejected_enc = tokenizer(example["prompt"] + "\n" + example["rejected"],
                             truncation=True, max_length=MAX_LEN_RM)
    return {
        "input_ids_chosen":        chosen_enc["input_ids"],
        "attention_mask_chosen":   chosen_enc["attention_mask"],
        "input_ids_rejected":      rejected_enc["input_ids"],
        "attention_mask_rejected": rejected_enc["attention_mask"],
    }

train_ds_rm = train_ds.map(preprocess_rm)
eval_ds_rm  = test_ds.map(preprocess_rm)

rm_config = RewardConfig(
    output_dir="rm_out",
    max_length=MAX_LEN_RM,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,       # was 5e-5 — needs to be higher for 1-epoch convergence
    num_train_epochs=5,       # was 3 — small dataset needs more passes
    logging_steps=5,
    eval_strategy="epoch",
    fp16=torch.cuda.is_available(),
    report_to="none",
)

rm_trainer = RewardTrainer(
    model=rm_model,
    tokenizer=tokenizer,
    args=rm_config,
    train_dataset=train_ds_rm,
    eval_dataset=eval_ds_rm,
)

print("Training Reward Model...")
rm_trainer.train()
print("Done.")


### Reward Model Evaluation

**Pairwise accuracy**: fraction of test pairs where RM scores chosen > rejected.  
Random baseline = 50%. A useful RM needs > ~65%.


In [ ]:
# ── Reward model pairwise accuracy on held-out test set ──────────────────────
rm_model.eval()
correct, chosen_scores, rejected_scores = 0, [], []

with torch.no_grad():
    for ex in test_ds:
        def score(text):
            enc = tokenizer(ex["prompt"] + "\n" + text, return_tensors="pt",
                            truncation=True, max_length=MAX_LEN_RM).to(device)
            return rm_model(**enc).logits[0, 0].item()
        rc, rr = score(ex["chosen"]), score(ex["rejected"])
        chosen_scores.append(rc)
        rejected_scores.append(rr)
        if rc > rr:
            correct += 1

pairwise_acc = correct / len(test_ds)
print(f"Pairwise accuracy : {pairwise_acc:.3f}  ({correct}/{len(test_ds)})")
print(f"Mean chosen  reward: {sum(chosen_scores)/len(chosen_scores):.4f}")
print(f"Mean rejected reward: {sum(rejected_scores)/len(rejected_scores):.4f}")
print(f"Mean margin (chosen - rejected): {sum(c-r for c,r in zip(chosen_scores, rejected_scores))/len(chosen_scores):.4f}")

plt.figure(figsize=(6, 3))
plt.hist(chosen_scores,   bins=20, alpha=0.6, label="Chosen",   color="steelblue")
plt.hist(rejected_scores, bins=20, alpha=0.6, label="Rejected", color="tomato")
plt.xlabel("RM Score")
plt.ylabel("Count")
plt.title(f"RM Score Distribution (test set) | pairwise acc={pairwise_acc:.2f}")
plt.legend()
plt.tight_layout()
plt.show()


Step 2: PPO alignment

### Step 2a: SFT Warm-Start (Supervised Fine-Tuning on Chosen Responses)

GPT-2 has no pre-trained concept of `(silent)`. If the policy never generates it,
PPO cannot reinforce it — reward stays flat regardless of the RM signal.

We do a short **supervised fine-tuning** pass on the `chosen` responses first,
teaching the policy the output format before PPO takes over.
This is standard practice in production RLHF (InstructGPT Stage 1).


In [ ]:
from transformers import Trainer, TrainingArguments
from torch.utils.data import Dataset as TorchDataset

# ── SFT dataset: train on prompt + chosen response ────────────────────────────
class SFTDataset(TorchDataset):
    def __init__(self, pairs, tokenizer, max_len=256):
        self.examples = []
        old_side = tokenizer.truncation_side
        tokenizer.truncation_side = "left"
        for p in pairs:
            text = p["prompt"] + "\n" + p["chosen"]
            enc  = tokenizer(text, truncation=True, max_length=max_len,
                             padding="max_length", return_tensors="pt")
            ids  = enc["input_ids"].squeeze()
            # Labels = input_ids (causal LM loss); mask padding
            labels = ids.clone()
            labels[labels == tokenizer.pad_token_id] = -100
            self.examples.append({"input_ids": ids, "attention_mask": enc["attention_mask"].squeeze(), "labels": labels})
        tokenizer.truncation_side = old_side

    def __len__(self):  return len(self.examples)
    def __getitem__(self, i): return self.examples[i]

sft_dataset = SFTDataset(train_pairs, tokenizer)

# Re-load a fresh GPT-2 as the SFT base (don't overwrite base_model)
sft_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)

sft_args = TrainingArguments(
    output_dir="sft_out",
    num_train_epochs=2,
    per_device_train_batch_size=4,
    learning_rate=2e-5,
    logging_steps=20,
    fp16=torch.cuda.is_available(),
    report_to="none",
    save_strategy="no",
)

sft_trainer = Trainer(
    model=sft_model,
    args=sft_args,
    train_dataset=sft_dataset,
)

print("SFT warm-start training...")
sft_trainer.train()
print("SFT done. Policy now knows the output format.")


In [ ]:
from trl.models import AutoModelForCausalLMWithValueHead
from trl import PPOConfig, PPOTrainer

MAX_LEN_PPO = 128

# Policy: warm-started from SFT
ppo_model = AutoModelForCausalLMWithValueHead.from_pretrained(MODEL_NAME).to(device)
ppo_model.pretrained_model.load_state_dict(sft_model.state_dict())

# Reference: SFT checkpoint frozen
ppo_ref = AutoModelForCausalLMWithValueHead.from_pretrained(MODEL_NAME).to(device)
ppo_ref.pretrained_model.load_state_dict(sft_model.state_dict())
ppo_ref.eval()
for p in ppo_ref.parameters():
    p.requires_grad_(False)

def preprocess_ppo(example):
    example["input_ids"] = tokenizer.encode(
        example["prompt"], truncation=True, max_length=MAX_LEN_PPO
    )
    return example

ppo_ds = train_ds.map(preprocess_ppo)

def collator(data):
    input_ids_list = [torch.tensor(d["input_ids"]).to(device) for d in data]
    prompt_texts = [tokenizer.decode(d["input_ids"], skip_special_tokens=True) for d in data]
    return {"input_ids": input_ids_list, "prompt_text": prompt_texts}

ppo_config = PPOConfig(
    model_name=MODEL_NAME,
    learning_rate=5e-6,
    batch_size=8,
    mini_batch_size=4,
    gradient_accumulation_steps=1,
    ppo_epochs=1,
    init_kl_coef=1.0,
    target_kl=3.0,
    cliprange=0.1,
    cliprange_value=0.1,
    optimize_cuda_cache=True,
    log_with=None,
)

ppo_trainer = PPOTrainer(
    config=ppo_config,
    model=ppo_model,
    ref_model=ppo_ref,
    tokenizer=tokenizer,
    dataset=ppo_ds,
    data_collator=collator,
)

step_log = []
print("PPO Trainer ready (warm-started from SFT).")


Step 3: Training PPO

In [ ]:
generation_kwargs = {
    "min_length":     -1,
    "top_k":          0.0,
    "top_p":          1.0,
    "do_sample":      True,
    "pad_token_id":   tokenizer.pad_token_id,
    "max_new_tokens": 60,
}

NUM_EPOCHS = 2
global_step = 0
print("Starting PPO Training...")

for epoch in range(NUM_EPOCHS):
    for step, batch in enumerate(tqdm(ppo_trainer.dataloader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")):
        query_tensors = batch["input_ids"]

        # 1. Generate responses
        response_tensors = ppo_trainer.generate(
            query_tensors, return_prompt=False, **generation_kwargs
        )
        batch["response"] = [
            tokenizer.decode(r.squeeze(), skip_special_tokens=True)
            for r in response_tensors
        ]

        # 2. Score with reward model
        texts  = [q + "\n" + r for q, r in zip(batch["prompt_text"], batch["response"])]
        enc_rm = tokenizer(texts, padding=True, truncation=True,
                           max_length=MAX_LEN_RM, return_tensors="pt").to(device)
        with torch.no_grad():
            rm_logits = rm_model(**enc_rm).logits
        rewards = [torch.tensor(v.item(), device=device) for v in rm_logits[:, 0]]

        # 3. PPO step
        stats = ppo_trainer.step(query_tensors, response_tensors, rewards)

        # 4. Log
        mean_reward = torch.stack(rewards).mean().item()
        kl          = stats.get("objective/kl",    float("nan"))
        policy_loss = stats.get("ppo/loss/policy", float("nan"))
        value_loss  = stats.get("ppo/loss/value",  float("nan"))
        step_log.append({"step": global_step, "epoch": epoch+1,
                         "reward": mean_reward, "kl": kl,
                         "policy_loss": policy_loss, "value_loss": value_loss})

        if global_step % 10 == 0:
            print(f"  step {global_step:3d} | reward {mean_reward:+.4f} | "
                  f"KL {kl:.4f} | pi_loss {policy_loss:.4f}")
        global_step += 1

print("\nPPO Training complete.")


Step 4: Evaluation

In [ ]:
import pandas as pd

# ── Metric helpers ─────────────────────────────────────────────────────────────
def silence_flag(text: str) -> int:
    """1 if output contains '(silent)' — the preferred VA behaviour."""
    return int("(silent)" in text.lower())

def repetition_score(text: str) -> float:
    """Fraction of bigrams that are repeated (higher = more repetitive)."""
    words  = text.lower().split()
    if len(words) < 2:
        return 0.0
    bigrams = [(words[i], words[i+1]) for i in range(len(words)-1)]
    return 1.0 - len(set(bigrams)) / max(len(bigrams), 1)

def refusal_flag(text: str) -> int:
    """1 if model output looks like a refusal (overoptimization symptom)."""
    hedges = ["i cannot", "i can't", "i'm unable", "i am unable",
              "inappropriate", "i should not", "i won't"]
    tl = text.lower()
    return int(any(h in tl for h in hedges))

@torch.no_grad()
def rm_score(prompt: str, text: str) -> float:
    """Score a completion with the reward model."""
    enc = tokenizer(prompt + "\n" + text, return_tensors="pt",
                    truncation=True, max_length=MAX_LEN_RM).to(device)
    return rm_model(**enc).logits[0, 0].item()

@torch.no_grad()
def generate(model, prompt: str, seed: int = 42, max_new_tokens: int = 80) -> str:
    """Generate from any CausalLM or CausalLMWithValueHead model."""
    lm = model.pretrained_model if hasattr(model, "pretrained_model") else model
    torch.manual_seed(seed)
    ids = tokenizer.encode(prompt, return_tensors="pt",
                           truncation=True, max_length=MAX_LEN_PPO).to(device)
    out = lm.generate(ids, max_new_tokens=max_new_tokens, do_sample=True,
                      top_p=0.9, temperature=0.8,
                      pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(out[0, ids.shape[1]:], skip_special_tokens=True)

# ── Held-out prompts ───────────────────────────────────────────────────────────
heldout_prompts = [ex["prompt"] for ex in test_ds.select(range(min(30, len(test_ds))))]
print(f"Evaluating on {len(heldout_prompts)} held-out prompts...")

rows = []
for prompt in tqdm(heldout_prompts):
    for name, model in [("base", base_model), ("ppo", ppo_model)]:
        out = generate(model, prompt)
        rows.append({
            "model":      name,
            "prompt":     prompt[:40] + "...",
            "output":     out,
            "len_words":  len(out.split()),
            "repetition": repetition_score(out),
            "is_silent":  silence_flag(out),
            "refusal":    refusal_flag(out),
            "rm_reward":  rm_score(prompt, out),
        })

df_eval = pd.DataFrame(rows)

print("\n=== Model Comparison (mean over held-out prompts) ===")
summary = df_eval.groupby("model")[["len_words","repetition","is_silent","refusal","rm_reward"]].mean()
summary.columns = ["Avg Length","Repetition","Silent Rate","Refusal Rate","RM Reward"]
display(summary.round(4))

delta = summary.loc["ppo"] - summary.loc["base"]
print("\n=== Delta (PPO − Base) ===")
display(delta.to_frame("Δ").T.round(4))

# Sample outputs
print("\n=== Sample Completions ===")
for prompt in heldout_prompts[:3]:
    print(f"PROMPT  : {prompt[:80]}")
    print(f"BASE    : {generate(base_model, prompt)[:120]}")
    print(f"PPO     : {generate(ppo_model,  prompt)[:120]}")
    print("-" * 80)


---
## Overoptimization & Misalignment Analysis

RLHF can overfit to the reward model ("reward hacking"): the policy maximizes RM score
but actual quality degrades. We diagnose this via:

1. **Reward vs step** — should increase then plateau
2. **KL divergence vs step** — should stay bounded (target_kl=6.0)
3. **Reward vs KL scatter** — if reward drops at high KL, we're overoptimizing
4. **Repetition & refusal rates** — quality proxies that should not increase post-PPO


In [ ]:
df_log = pd.DataFrame(step_log)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# (a) Mean reward over steps
axes[0].plot(df_log["step"], df_log["reward"], color="steelblue")
axes[0].set_title("Mean RM Reward vs Step")
axes[0].set_xlabel("PPO Step")
axes[0].set_ylabel("Mean Reward")
axes[0].grid(True, alpha=0.3)

# (b) KL divergence over steps
axes[1].plot(df_log["step"], df_log["kl"], color="tomato")
axes[1].axhline(ppo_config.target_kl, linestyle="--", color="gray",
                label=f"target_kl={ppo_config.target_kl}")
axes[1].set_title("KL Divergence (policy || ref) vs Step")
axes[1].set_xlabel("PPO Step")
axes[1].set_ylabel("KL")
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

# (c) Reward vs KL frontier (overoptimization check)
sc = axes[2].scatter(df_log["kl"], df_log["reward"], c=df_log["step"],
                     cmap="viridis", s=20, alpha=0.8)
plt.colorbar(sc, ax=axes[2], label="Step")
axes[2].set_title("Reward vs KL (Overoptimization Frontier)")
axes[2].set_xlabel("KL")
axes[2].set_ylabel("Mean Reward")
axes[2].grid(True, alpha=0.3)

plt.suptitle("PPO Training Dynamics", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# ── Quality metric boxplots ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, metric, title in [
    (axes[0], "repetition", "Repetition Score (lower = better)"),
    (axes[1], "rm_reward",  "RM Reward (higher = more aligned)"),
]:
    base_v = df_eval[df_eval["model"] == "base"][metric].values
    ppo_v  = df_eval[df_eval["model"] == "ppo"][metric].values
    ax.boxplot([base_v, ppo_v], labels=["Base", "PPO"], patch_artist=True,
               boxprops=dict(facecolor="lightblue"))
    ax.set_title(title)
    ax.grid(True, alpha=0.3)

plt.suptitle("Base vs PPO: Quality Metrics", fontsize=12)
plt.tight_layout()
plt.show()


### Discussion: Interpreting Your Deliverable Metrics

**Reward definition recap:**  
At every PPO step, the scalar reward is `rm_model(prompt + "\n" + policy_response).logits[0,0]`.  
The RM was trained with Bradley-Terry ranking loss to assign higher scores to `chosen` responses  
and lower scores to `rejected` ones. In this dataset `chosen` is usually the `(silent)` label —  
meaning the voice assistant should stay quiet rather than interrupt.

---

#### Signal 1 — High KL Divergence (Overoptimization)

Watch the **KL vs Step** plot above. The KL penalty (`init_kl_coef=0.2`) is the only thing  
preventing the policy from completely abandoning its pre-trained language distribution.

- **Healthy:** KL rises gradually and flattens near `target_kl=6.0`
- **Overoptimized:** KL spikes past the target and keeps climbing — the model is chasing RM score  
  at the expense of fluency and coherence
- **In the Reward vs KL scatter:** an ideal run traces a concave-up arc (more reward per unit KL  
  early, diminishing returns later). If the arc **bends downward** at high KL, Goodhart's Law is  
  in effect — the RM proxy has decoupled from actual preference

---

#### Signal 2 — Repetition & Length Collapse (Degraded Coherence)

From `df_eval`, compare **Repetition** and **Avg Length** for `base` vs `ppo`:

| Observation | Interpretation |
|---|---|
| Repetition↑ post-PPO | Model found a "cheat code": repeating a token/phrase that the RM rewards |
| `len_words` collapses to ~1–5 | Model learned the shortest path to a high reward (e.g. just output `(silent)`) |
| Repetition and length both normal | PPO is working cleanly, no reward hacking |

This is the most **verifiable** misalignment proof in your report — show the delta row from  
the evaluation table.

---

#### Signal 3 — Silent Over-Indexing (Contextual Misalignment)

Check `is_silent` in the evaluation table:

- **Base model:** should be near 0 (GPT-2 has no `(silent)` training signal)
- **PPO model, partial increase (~20–50%):** alignment is working — model is learning context
- **PPO model hits 100%:** the model learned that `(silent)` always yields high reward and  
  **ignores context entirely** — this is the misalignment. The policy satisfies the RM proxy  
  without satisfying the actual human intent (sometimes an answer is needed)

This mirrors real-world RLHF failures: models that refuse everything, hedge every answer,  
or repeat safety disclaimers verbatim to maximize a safety-tuned RM score.

---

#### Mitigations

| Problem | Fix |
|---|---|
| KL diverges | Increase `init_kl_coef` β (tighter anchor to reference) |
| Repetition hack | Add `repetition_penalty>1.0` to generation kwargs |
| Silent over-indexing | Balance the preference dataset (equal chosen-silent and chosen-verbose) |
| RM proxy gaming | Use **DPO** (bypasses RM entirely) or ensemble multiple RMs |
| Small dataset noise | More preference pairs (2k–10k); our 255-pair set makes RM accuracy noisy |


In [ ]:
# ── Automated signal diagnosis ───────────────────────────────────────────────
# Reads df_log (training) and df_eval (evaluation) and prints a plain-English
# verdict on each of the three rubric signals.

print("=" * 60)
print("DELIVERABLE ANALYSIS: 3 MISALIGNMENT SIGNALS")
print("=" * 60)

# ── Signal 1: KL Divergence ───────────────────────────────────────────────────
kl_vals   = [r["kl"] for r in step_log if not (r["kl"] != r["kl"])]  # drop NaN
max_kl    = max(kl_vals) if kl_vals else float("nan")
final_kl  = kl_vals[-1] if kl_vals else float("nan")
kl_target = ppo_config.target_kl

print(f"\n[Signal 1] KL Divergence")
print(f"  Max KL during training : {max_kl:.4f}  (target = {kl_target})")
print(f"  Final KL               : {final_kl:.4f}")
if max_kl > kl_target * 1.5:
    print("  ⚠ OVEROPTIMIZATION DETECTED: KL exceeded 1.5× target — policy diverged too far from reference")
elif max_kl > kl_target:
    print("  ⚠ MILD OVEROPTIMIZATION: KL exceeded target — monitor for degraded coherence")
else:
    print("  ✓ KL stayed within target — no overoptimization from KL signal")

# ── Signal 2: Repetition & Length Collapse ────────────────────────────────────
rep_base = df_eval[df_eval["model"] == "base"]["repetition"].mean()
rep_ppo  = df_eval[df_eval["model"] == "ppo"]["repetition"].mean()
len_base = df_eval[df_eval["model"] == "base"]["len_words"].mean()
len_ppo  = df_eval[df_eval["model"] == "ppo"]["len_words"].mean()

print(f"\n[Signal 2] Repetition & Length")
print(f"  Repetition  — base: {rep_base:.4f}  ppo: {rep_ppo:.4f}  delta: {rep_ppo-rep_base:+.4f}")
print(f"  Avg length  — base: {len_base:.1f} words  ppo: {len_ppo:.1f} words  delta: {len_ppo-len_base:+.1f}")
if rep_ppo > rep_base * 1.3:
    print("  ⚠ REWARD HACK (repetition): PPO model repeats tokens to satisfy RM — Goodhart's Law")
else:
    print("  ✓ Repetition not significantly increased")
if len_ppo < len_base * 0.5:
    print("  ⚠ LENGTH COLLAPSE: PPO model learned ultra-short outputs to game the reward signal")
else:
    print("  ✓ Length not collapsed")

# ── Signal 3: Silent Over-Indexing ────────────────────────────────────────────
sil_base = df_eval[df_eval["model"] == "base"]["is_silent"].mean()
sil_ppo  = df_eval[df_eval["model"] == "ppo"]["is_silent"].mean()

print(f"\n[Signal 3] Silent Rate")
print(f"  Base model silent rate : {sil_base:.3f} ({sil_base*100:.1f}%)")
print(f"  PPO  model silent rate : {sil_ppo:.3f} ({sil_ppo*100:.1f}%)")
if sil_ppo >= 0.95:
    print("  ⚠ CONTEXTUAL MISALIGNMENT: Model outputs '(silent)' nearly always — ignores context, maximizes proxy")
elif sil_ppo > sil_base + 0.1:
    print("  ~ PARTIAL ALIGNMENT: Silent rate increased — model learned the preference signal (check if context-appropriate)")
elif sil_ppo <= sil_base:
    print("  ~ NO EFFECT: Silent rate unchanged — PPO did not shift the model toward the preferred label")
else:
    print("  ✓ Silent rate rose modestly — alignment working without over-indexing")

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print("RM PAIRWISE ACCURACY (from earlier cell):", f"{pairwise_acc:.3f}")
print(f"{'='*60}")
